# 데모 2 — 센서 데이터부터 LLM 해설까지 전체 파이프라인

운영 중인 시스템은 Kafka 로 센서를 받지만, 이 노트북은 **브로커 없이 CSV 로 같은 경로를 재현**한다.
각 단계는 `server.py` 가 실제로 쓰는 코드를 그대로 호출한다.

```
data/*.csv  ──[1]──▶  3-소스 병합 (218 features)
                            │
                          [2]  InferenceEngine  (AutoEncoder MSE + LightGBM 분류)
                            │  run 전 구간 중 Peak MSE 시점 포착
                            │
                          [3]  SHAPExplainer    (TreeExplainer → 기여 센서 Top-5)
                            │
              ┌─────────────┴─────────────┐
            [4] SHAPAgent               [5] GraphRAGAgentV2
                OpenAI LLM                  Neo4j KG → OpenAI LLM
                원인 해설                    정비 SOP 권고
```

| 단계 | 재현하는 실제 코드 |
|---|---|
| [1] | `kafka_streamer.py` 의 메타 컬럼 제외 + `worker.py` 의 3-소스 `dict.update` 병합 |
| [2] | `server.py:71` 과 같은 인자로 `InferenceEngine` 생성, Phase 2 의 Peak MSE 버퍼 |
| [3] | `server.py:544-548` 의 `pred_idx` 결정 → `scaler.transform` → `explain` |
| [4] | `server.py:632` `_call_shap_agent()` |
| [5] | `server.py:720` `_call_rag_v2()` |

In [1]:
"""[0] 환경 준비 — 프로젝트 루트로 이동하고, 이 프로세스에서만 LLM 호출을 허용한다."""
import os, sys, time, json, logging, warnings
from pathlib import Path

# 출력이 읽히도록 잡음만 줄인다(동작에는 영향 없음).
#   - sklearn: DataFrame 대신 ndarray 를 넘길 때 나오는 feature-name 경고가 스텝마다 반복된다
#   - httpx  : OpenAI 요청마다 INFO 로그를 stderr 로 찍는다
warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)

# 노트북은 demo/ 에 있지만 프로젝트 코드는 루트 기준 상대경로(models/, data/)를 쓴다.
ROOT = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# .env 의 OPENAI_ENABLED=false 는 24/7 워커의 과금을 막는 의도된 설정이다.
# load_dotenv() 는 이미 설정된 환경변수를 덮어쓰지 않으므로, 먼저 켜 두면
# .env 파일을 건드리지 않고 이 프로세스에서만 LLM 호출을 열 수 있다.
os.environ["OPENAI_ENABLED"] = "true"

from dotenv import load_dotenv
load_dotenv()

# .env 의 키 이름이 OPEN_AI_API_KEY 라서 SDK 가 찾는 OPENAI_API_KEY 로 옮긴다.
# (agents/shap_agent.py 11~12 행과 동일한 처리)
if "OPEN_AI_API_KEY" in os.environ and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = os.environ["OPEN_AI_API_KEY"]

MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
_key = os.environ.get("OPENAI_API_KEY", "")

print(f"project root   : {ROOT.name}/")
print(f"python         : {sys.version.split()[0]}")
print(f"model          : {MODEL}")
print(f"api key        : {_key[:7]}...{_key[-4:]} (len={len(_key)})")
print(f"OPENAI_ENABLED : {os.getenv('OPENAI_ENABLED')}   <- this process only; .env stays false")


project root   : etch_proj_final/
python         : 3.12.10
model          : gpt-4o-mini
api key        : sk-proj...MFIA (len=164)
OPENAI_ENABLED : true   <- this process only; .env stays false


## [1/5] 3-소스 병합

공장에서는 OES · MACHINE · RFM 세 계측 시스템이 각자 Kafka 토픽으로 보낸다.
`worker.py` 는 `(equipment_id, run_id, time_step)` 키로 셋을 모아 하나의 센서 딕셔너리로 합친다.

여기서는 같은 규칙을 CSV 에 적용한다 — 세 파일의 `Time_Step` 교집합만 쓰고,
메타 컬럼을 뺀 나머지를 `dict.update` 로 합친다.

In [2]:
"""[1/5] data/*.csv 3개를 읽어 한 시점 = 한 벡터로 병합"""
import pandas as pd, numpy as np

TARGET_FAULT = "TCP +30"          # LightGBM 라벨 표기 (KG 쪽은 "TCP+30")

t0 = time.perf_counter()
oes = pd.read_csv("data/OES_integrated.csv")
mac = pd.read_csv("data/MACHINE_integrated.csv")
rfm = pd.read_csv("data/RFM_integrated.csv")
for _df in (oes, mac, rfm):
    _df.columns = _df.columns.str.strip()

RUN = mac.loc[mac.Fault_Name == TARGET_FAULT, "Run_Name"].iloc[0]
o = oes[oes.Run_Name == RUN]
m = mac[mac.Run_Name == RUN]
r = rfm[rfm.Run_Name == RUN]

# 세 소스가 모두 도착한 시점만 추론에 쓴다 (worker.py 의 merge buffer 와 같은 조건)
steps = sorted(set(o.Time_Step) & set(m.Time_Step) & set(r.Time_Step))

# 센서값이 아닌 메타 컬럼은 제외한다 (kafka_streamer.py 와 동일)
META = {"Data_Type", "Run_Name", "Fault_Name", "Time_Step",
        "Time", "TIME", "Step Number", "Is_Synthetic", "Synthesis_Method"}

def merge_at(step):
    """한 time_step 의 OES+MACHINE+RFM 을 하나의 dict 로 합친다."""
    merged = {}
    for df in (o, m, r):
        row = df[df.Time_Step == step].iloc[0].to_dict()
        merged.update({k: v for k, v in row.items() if k not in META})
    return merged

sample = merge_at(steps[0])
print(f"run            : {RUN}   (ground truth = {TARGET_FAULT})")
print(f"common steps   : {len(steps)}  (OES {len(o)} / MACHINE {len(m)} / RFM {len(r)} rows)")
print(f"merged features: {len(sample)}")
print(f"elapsed        : {time.perf_counter()-t0:.2f} s")
print()
print("sample (first 6 of merged vector)")
for k in list(sample)[:6]:
    print(f"   {k:<16} {sample[k]}")

run            : 3120   (ground truth = TCP +30)
common steps   : 28  (OES 38 / MACHINE 100 / RFM 28 rows)
merged features: 218
elapsed        : 0.48 s

sample (first 6 of merged vector)
   250.0            1884.4
   261.8            24338.4
   266.6            2619.2
   272.2            80318.6
   278.3            8300.2
   284.6            1678.5


## [2/5] 추론 — AutoEncoder + LightGBM

`InferenceEngine` 은 두 단계로 판정한다.

1. **AutoEncoder** — 218차원을 16차원으로 압축했다 복원해 재구성 오차(MSE)를 낸다. MSE 가 임계치를 넘으면 이상.
2. **LightGBM** — 이상이면 16개 결함 클래스 확률을 낸다.

운영 시스템은 run 이 도는 내내 **MSE 가 가장 컸던 시점**을 버퍼에 들고 있다가
run 이 끝나면 그 시점으로 심층 분석(Phase 2)을 돌린다. 여기서도 같은 방식으로 Peak 를 찾는다.

In [3]:
"""[2/5] run 전 구간 추론 → Peak MSE 시점 포착 (server.py Phase 2 와 동일)"""
from inference import InferenceEngine

logging.getLogger("InferenceEngine").setLevel(logging.WARNING)   # 스텝별 로그 억제

engine = InferenceEngine(lgbm_confidence_threshold=0.8)          # server.py:71 과 같은 인자
print(f"features       : {len(engine.features)}")
print(f"AE threshold   : {engine.base_threshold:.4f}  (suspect {engine.suspect_threshold:.4f})")
print(f"LGBM classes   : {len(engine.le.classes_)}")
print()

t0 = time.perf_counter()
peak_res, peak_metrics, peak_step, n_anom = None, None, None, 0
for s in steps:
    metrics = merge_at(s)
    res = engine.predict(metrics)
    n_anom += bool(res["is_anomaly"])
    if peak_res is None or res["mse"] > peak_res["mse"]:
        peak_res, peak_metrics, peak_step = res, metrics, s
infer_sec = time.perf_counter() - t0

print(f"inferred {len(steps)} steps in {infer_sec:.2f} s "
      f"({infer_sec/len(steps)*1000:.1f} ms/step) - anomalies {n_anom}/{len(steps)}")
print()
print(f"PEAK MSE at time_step {peak_step}")
print(f"   mse         : {peak_res['mse']:.4f}   (threshold {peak_res['current_threshold']:.4f})")
print(f"   is_anomaly  : {peak_res['is_anomaly']}")
print(f"   status      : {peak_res['status']}")
print(f"   confidence  : {peak_res['confidence']:.3f}")
print(f"   top-3 candidates")
for c in peak_res["top_candidates"]:
    print(f"      {c['label']:<12} {c['confidence']:.3f}")

features       : 218
AE threshold   : 0.7509  (suspect 0.6021)
LGBM classes   : 16



inferred 28 steps in 3.70 s (132.0 ms/step) - anomalies 28/28

PEAK MSE at time_step 16
   mse         : 3.4422   (threshold 0.7509)
   is_anomaly  : True
   status      : TCP +30
   confidence  : 1.000
   top-3 candidates
      TCP +30      1.000
      RF +10       0.000
      Cl2 +5       0.000


## [3/5] SHAP — 어느 센서가 그 판정을 만들었나

`server.py:544-548` 의 경로를 그대로 따른다.
예측이 `Normal` 이나 `UNKNOWN FAULT` 로 나온 경우에는 Top-1 후보 라벨로 SHAP 을 계산한다.

In [4]:
"""[3/5] TreeExplainer 로 기여 센서 Top-5 산출 (server.py:544-548)"""
from shap_analysis import SHAPExplainer

explainer = SHAPExplainer(engine.lgb_model, engine.features)

target_label = peak_res["predicted_label"]
if target_label in ("Normal", "UNKNOWN FAULT") and peak_res["top_candidates"]:
    target_label = peak_res["top_candidates"][0]["label"]
pred_idx = list(engine.le.classes_).index(target_label)

m_df = pd.DataFrame([peak_metrics])
m_df.columns = m_df.columns.str.strip()
scaled = engine.scaler.transform(m_df[engine.features])

t0 = time.perf_counter()
analysis_data = explainer.explain(scaled, peak_metrics, pred_idx)
shap_sec = time.perf_counter() - t0

print(f"SHAP target label : {target_label}  (class index {pred_idx})")
print(f"elapsed           : {shap_sec:.2f} s")
print()
print(f"{'#':<3}{'sensor':<16}{'shap':>9}{'current':>12}{'mean':>12}{'status':>9}")
print("-" * 62)
for i, a in enumerate(analysis_data, 1):
    print(f"{i:<3}{a['sensor']:<16}{a['shap_value']:>+9.3f}"
          f"{a['current_value']:>12.4f}{a['mean_value']:>12.4f}{a['status']:>9}")

SHAP target label : TCP +30  (class index 13)
elapsed           : 0.06 s

#  sensor               shap     current        mean   status
--------------------------------------------------------------
1  S2P4               +4.745    -27.1200    -75.3130   Normal
2  S34I3              +3.243      0.0299      0.0177     High
3  Endpt A            +1.861   3390.0000   1278.7639   Normal
4  S1I3               +1.234      0.0030      0.0028     High
5  364.33             +1.131    569.3000     80.3328     High


In [5]:
"""[3.5] SHAP 기여도 막대 그래프 — 프론트엔드 화면 3 과 같은 그림"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

names = [a["sensor"] for a in analysis_data][::-1]
vals  = [a["shap_value"] for a in analysis_data][::-1]
cols  = ["#e15759" if v > 0 else "#4e79a7" for v in vals]

fig, ax = plt.subplots(figsize=(8, 3.6), dpi=150)
ax.barh(names, vals, color=cols, height=0.62)
ax.axvline(0, color="#888", lw=0.8)
ax.set_xlabel("SHAP value  (기여도)")
ax.set_title(f"{RUN}  step {peak_step}  |  예측: {target_label}  |  MSE {peak_res['mse']:.3f}")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for y, v in enumerate(vals):
    ax.text(v + (0.12 if v > 0 else -0.12), y, f"{v:+.2f}",
            va="center", ha="left" if v > 0 else "right", fontsize=8)
ax.margins(x=0.16)
fig.tight_layout()

out = Path("docs/images"); out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "demo2-shap-chart.png", bbox_inches="tight", facecolor="white")
print(f"saved -> {out / 'demo2-shap-chart.png'}")
plt.close(fig)

saved -> docs\images\demo2-shap-chart.png


## [4/5] LLM 원인 해설 — `SHAPAgent`

여기가 **첫 번째 LLM API 호출**이다.
숫자로만 있던 SHAP 결과를 공정 엔지니어가 읽을 수 있는 한국어 기술 해설로 바꾼다.

In [6]:
"""[4/5] SHAP 결과 → OpenAI LLM → 한국어 원인 분석 (server.py:632)"""
from agents.shap_agent import SHAPAgent

display_fault = peak_res["status"] if peak_res["status"] != "UNKNOWN FAULT" else "감지된 결함"

t0 = time.perf_counter()
explanation = SHAPAgent().explain_fault(display_fault, analysis_data)
llm_sec = time.perf_counter() - t0

print(f"[SHAPAgent]  fault='{display_fault}'  {llm_sec:.2f} s\n")
print(explanation)

[SHAPAgent]  fault='TCP +30'  13.91 s

결함에 대한 센서 분석 결과를 바탕으로 기술적 분석을 제공합니다. 이번 결함은 TCP +30으로 감지되었으며, 여러 센서의 상태가 결함 발생에 기여하고 있습니다. 각 센서의 현재 상태와 SHAP 값을 통해 결함의 원인을 분석하겠습니다.

1. **S2P4 센서**
   - **현재 값**: -27.12
   - **평균 값**: -75.313
   - **상태**: 정상
   - **SHAP 방향**: 긍정적 영향
   - **분석**: S2P4 센서는 현재 정상 범위 내에 있으며, SHAP 값이 긍정적 영향을 미치고 있습니다. 이는 이 센서가 결함에 직접적인 영향을 미치지 않음을 시사합니다. 그러나, 이 센서의 값이 정상 범위에 있더라도, 다른 센서와의 상호작용을 통해 결함에 간접적으로 기여할 수 있습니다.

2. **S34I3 센서**
   - **현재 값**: 0.0299
   - **평균 값**: 0.0177
   - **상태**: 높음
   - **SHAP 방향**: 긍정적 영향
   - **분석**: S34I3 센서의 현재 값이 평균보다 높으며, 이는 장비의 특정 파라미터가 비정상적으로 증가하고 있음을 나타냅니다. 이 센서는 플라즈마 에칭 공정에서 중요한 역할을 하며, 높은 값은 가스 흐름이나 압력의 이상을 시사할 수 있습니다. 따라서, 이 센서의 높은 값은 가스 공급 시스템의 문제, 예를 들어 밸브의 고장이나 가스 유량 조절의 실패를 나타낼 수 있습니다.

3. **Endpt A 센서**
   - **현재 값**: 3390.0
   - **평균 값**: 1278.7639
   - **상태**: 정상
   - **SHAP 방향**: 긍정적 영향
   - **분석**: Endpt A 센서의 값은 정상 범위 내에 있지만, 평균보다 상당히 높은 수치를 기록하고 있습니다. 이는 공정의 특정 지점에서 비정상적인 조건이 발생하고 있음을 나타낼 수 있습니다. 이 센서는 공정의 최종 단계에서의 

## [5/5] GraphRAG — 지식그래프에서 정비 절차 찾기

**두 번째 LLM API 호출**. Neo4j 지식그래프를 먼저 조회하고, 그 결과를 근거로 LLM 이 정비 지침을 쓴다.

### V1 이 아니라 V2 를 쓰는 이유

현재 KG 의 관계 타입은 `CAUSED_BY` · `REMEDIATED_BY` · `DEGRADES` · `HAS_STEP` 이다.
V1(`agents/rag_agent.py`)의 Cypher 가 요구하는 `FIXED_BY` · `Task` · `Cause` 는 **그래프에 없다.**
그래서 V1 을 부르면 항상 "No specific SOP found" 가 나온다.
`agents/rag_agent_v2.py` 가 현재 스키마에 맞는 구현이다.

V2 는 두 단계로 검색한다.

- **Stage 1 (coarse)** — SHAP 상위 센서를 `Fault.sensors_low` / `sensors_high` 지문과 대조해 결함 후보를 매긴다
- **Stage 2 (fine)** — 상위 결함에서 `Fault → Mechanism → Component → SOP` 인과 체인을 따라간다

In [7]:
"""[5/5] GraphRAGAgentV2 — Neo4j 조회 + LLM 정비 지침 (server.py:_call_rag_v2)"""
from agents.rag_agent_v2 import GraphRAGAgentV2, parse_shap_to_lists, shap_list_to_dict

# SHAPExplainer 출력(list) → V2 가 받는 dict 로 변환.
# server.py 의 _call_rag_v2() 도 같은 어댑터를 쓴다 — 중복 구현 없음.
shap_for_rag = shap_list_to_dict(analysis_data)

rag = GraphRAGAgentV2()
try:
    # --- 매칭 진단: 지문 매칭이 왜 빗나가는지 먼저 드러낸다 ---
    low, high = parse_shap_to_lists(shap_for_rag)
    fp_hits = rag.stage1_match_fault(shap_for_rag, top_n=3)
    print("[stage 1] SHAP 지문 매칭")
    print(f"   low  : {low}")
    print(f"   high : {high}")
    print(f"   hits : {fp_hits if fp_hits else '없음'}")
    print("   -> 센서명은 SENSOR_ALIASES 로 KG 이름으로 정규화된다 (TCP Top Pwr -> TCP Top Power).")
    print()

    # KG 의 Fault 이름은 공백이 없다: 'TCP +30'(LGBM) vs 'TCP+30'(KG)
    hint = target_label.replace(" ", "")
    print(f"[fallback] 텍스트 힌트 = '{hint}'  (LGBM 라벨 '{target_label}' 에서 공백 제거 · 지문 매칭이 비었을 때만 쓰인다)")
    print()

    t0 = time.perf_counter()
    out = rag.recommend(
        shap_analysis=shap_for_rag,
        fault_name_hint=hint,
        question="이 이상 징후의 원인과 점검 절차를 알려주세요.",
    )
    rag_sec = time.perf_counter() - t0

    print(f"[stage 2] 인과 체인  ({rag_sec:.2f} s, context ~{out['token_estimate']} tokens)")
    for c in out["candidates"]:
        print(f"   candidate: {c.get('fault_id','?')} {c['name']}")
    chain = out["chain"]
    if chain:
        print(f"   fault     : {chain.get('fault_name')}  (ref: {chain.get('fault_ref')})")
        for mech in [x for x in chain.get("mechanisms", []) if x.get("mechanism")]:
            print(f"   mechanism : {mech['mechanism']}  ({mech.get('timescale')})")
        print(f"   components: {[c for c in chain.get('components', []) if c]}")
        for sop in [x for x in chain.get("sops", []) if x.get("sop_id")]:
            print(f"   SOP       : [{sop['sop_id']}] {sop['title']}")
    print()
    print("[LLM answer]")
    print(out["answer"])
finally:
    rag.close()

[stage 1] SHAP 지문 매칭
   low  : []
   high : ['S34I3', 'S1I3', '364.33']
   hits : 없음
   -> 센서명은 SENSOR_ALIASES 로 KG 이름으로 정규화된다 (TCP Top Pwr -> TCP Top Power).

[fallback] 텍스트 힌트 = 'TCP+30'  (LGBM 라벨 'TCP +30' 에서 공백 제거 · 지문 매칭이 비었을 때만 쓰인다)



[stage 2] 인과 체인  (4.79 s, context ~153 tokens)
   candidate: F10 TCP+30
   fault     : TCP+30  (ref: Wise 1999 Table 2)
   mechanism : Matcher Detune  (days)
   components: ['TCP Matcher', 'RF Matcher']
   SOP       : [SOP-TCP-RECAL] TCP Matcher Recalibration
   SOP       : [SOP-RF-RECAL] RF Bottom Matcher Recalibration

[LLM answer]
(1) 가장 가능성이 높은 고장 및 신뢰도: TCP+30 (신뢰도: 높음)

(2) 근본 원인 메커니즘: 매처 조정 불량 (시간 척도: 며칠; 증상: 반사 전력 상승, 임피던스 변화)

(3) 권장 SOP:
- 제목: TCP 매처 재조정
  초록: RF 케이블을 점검하고, 매처 커패시터를 확인한 후 조정 범위를 재조정합니다. 이 과정은 약 30분 소요됩니다.

(4) 가장 관련성이 높은 단계:
1. RF 케이블 점검: 손상이나 느슨함이 없는지 확인합니다.
2. 매처 커패시터 확인: 커패시터의 상태를 점검하고 필요시 교체합니다.
3. 조정 범위 재조정: 매처의 조정 범위를 재설정하여 정상 작동을 복구합니다.

참고: Wise 1999 Table 2 (KG 참조 필드)


## 요약

In [8]:
"""단계별 소요시간과 LLM 비용"""
PRICE_IN, PRICE_OUT = 0.150, 0.600      # gpt-4o-mini USD / 1M tokens (참고치)

est = [
    ("SHAPAgent",       len(json.dumps(analysis_data, ensure_ascii=False)) // 4 + 220,
                        len(explanation) // 2,   llm_sec),
    ("GraphRAG V2",     out["token_estimate"] + 180,
                        len(out["answer"]) // 2, rag_sec),
]

print("pipeline")
print(f"   [1] merge      {len(steps):>4} steps x 218 features")
print(f"   [2] inference  {infer_sec:>7.2f} s   peak MSE {peak_res['mse']:.4f} -> {peak_res['status']}")
print(f"   [3] SHAP       {shap_sec:>7.2f} s   top-1 {analysis_data[0]['sensor']}"
      f" ({analysis_data[0]['shap_value']:+.2f})")
print(f"   [4] LLM        {llm_sec:>7.2f} s")
print(f"   [5] GraphRAG   {rag_sec:>7.2f} s")
print()
print(f"{'llm call':<16}{'in':>8}{'out':>8}{'sec':>8}{'USD':>11}")
print("-" * 51)
total = 0.0
for name, i, o_, lat in est:
    cost = i / 1e6 * PRICE_IN + o_ / 1e6 * PRICE_OUT
    total += cost
    print(f"{name:<16}{i:>8}{o_:>8}{lat:>8.2f}{cost:>11.6f}")
print("-" * 51)
print(f"{'TOTAL (est.)':<16}{'':>8}{'':>8}{'':>8}{total:>11.6f}")
print()
print("OK - 센서 CSV 에서 LLM 정비 지침까지 전 구간 통과")

pipeline
   [1] merge        28 steps x 218 features
   [2] inference     3.70 s   peak MSE 3.4422 -> TCP +30
   [3] SHAP          0.06 s   top-1 S2P4 (+4.74)
   [4] LLM          13.91 s
   [5] GraphRAG      4.79 s

llm call              in     out     sec        USD
---------------------------------------------------
SHAPAgent            461     876   13.91   0.000595
GraphRAG V2          333     188    4.79   0.000163
---------------------------------------------------
TOTAL (est.)                               0.000758

OK - 센서 CSV 에서 LLM 정비 지침까지 전 구간 통과
